# Code Lab - An Entire RAG Pipeline

---

En esta sección vamos a crear un **RAG pipeline** completo y desde cero, que servirá como base para profundizar en los capítulos posteriores. En este capítulo se incluirá un pipeline con las siguientes características:

- Vincular un LLM con una cuenta de OpenAI
- Instalación de paquetes de Python
- Web crawling, división de documentos y embedding chunks para la indexación de datos
- Búsqueda por vectores similares (vector similarity search)
- Generar respuestas integrando contexto a los prompts
- Sin interfaz 

En primer lugar instalamos e importamos las librerías necesarias:

In [1]:
# %pip install langchain_community langchain_experimental langchain-ollama langchainhub chromadb langchain beautifulsoup4

In [2]:
import os
from langchain_community.document_loaders import WebBaseLoader
import bs4
import ollama
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_experimental.text_splitter import SemanticChunker

USER_AGENT environment variable not set, consider setting it to identify your requests.


---

### **1. Indexing**

Ahora viene la primera fase del RAG, **indexing**, donde obtendremos los datos del prompt del usuario, les haremos un pre-procesamiento y los vectorizaremos. En este pequeño apartado haremos lo siguiente:

- Web loading y web crawling
- Dividir (splitting) los datos en chunks para que el algoritmo de vectorización de *Chroma* sea más eficiente
- Convertir los chunks en embeddings
- Añadir los chunks y embeddings a la *vector store* de *Chroma*

#### 1.1 Web Loading y Web Crawling

El contenido lo vamos a sacar de la siguiente página:

In [3]:
webPage = 'https://lilianweng.github.io/posts/2023-06-23-agent/'

loader = WebBaseLoader(
    web_path = webPage,
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ('post-title', 'post-header', 'post-content')
        )
    ),
)

docs = loader.load()


De esta manera podemos obtener el contenido de una web como documentos.

*WebBaseLoader* hace mucho trabajo:

1. Petición HTTP a la URL que hemos especificado
2. Hace un parseo del HTML son *BeautifulSoup*, parseando únicamente los elementos incluidos en *parse_only*
3. Extrae el texto del parseo
4. Crea objetos tipo *document* con el contenido de la web que hemos extraído (que se guardan en la variable docs en este caso)

Una vez hecho esto pasamos al siguiente paso, splitting

#### 1.2 Splitting

En este paso simplemente vamos a dividir el documento que hemos obtenido anteriormente en distintos *chunks*, para reducir tiempo de procesamiento, convertiéndolos en textos mucho más manejables sin perder la coherencia de cada *chunk*. 

En nuestro caso vamos a usar *SemanticChunker* aunque hay otras muchas opciones:

In [4]:
embeddings = OllamaEmbeddings(model='nomic-embed-text')
text_splitter = SemanticChunker(embeddings)
splits = text_splitter.split_documents(docs)
print(len(splits))

21


Vemos que *SemanticChunker* nos divide el documento en 21 chunks. Esto se hace en función del contexto de cada chunk, en lugar de proporcionar una longitud establecida. Esto en general nos hace una mejor división del texto al ser semántica (por ello necesita un modelo como *OllamaEmbeddings*), pero es más pesado computacionalmente.

Vamos a probar a ver un Chunker diferente, el cual si se basa en tamaño concreto, conocido como *RecursiveCharacterTextSplitter*, uno de los splitters más usados de langchain.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter_size = RecursiveCharacterTextSplitter(
    chunk_size = 500,   # Tamaño máximo de cada chunk
    chunk_overlap = 50,   # Indica cuantos caracteres se repiten entre un chunk y el siguiente (para no perder contexto)
    separators = ['\n\n', '\n', ' ', '']   # Lista ordenada de los separadores (intenta separar por parrafos, si no cabe en chunk_size prueba por lineas, etc.)
)

splits_size = text_splitter_size.split_documents(docs)
print(len(splits_size))

134


Y vemos que el tiempo en este caso es mucho menor, ya que estamos indicando el tamaño de los chunks y la separación es mucho más simple, pero la calidad de los chunks baja considerablemente. 

Hemos que decidir entre calidad o eficiencia, aunque en este caso, vamos a optar por calidad ya que no hay mucho texto.

#### 1.3 Embeddings y Vector Store de Chroma

A continuación vamos a crear la *vector store* con *Chroma* y vamos a guardar los *embeddings* de nuestro texto. Esto lo podemos hacer muy sencillamente con *Chroma* en *Python*:

In [6]:
vector_store = Chroma.from_documents(
    documents = splits,
    embedding = OllamaEmbeddings(model='nomic-embed-text')
)

retriever = vector_store.as_retriever()

Internamente, el método *Chroma.from_documents()* está haciendo lo siguiente:

1. Itera sobre cada *Document* en la variable *splits*
2. Para cada *Document*, usa el embedding, en este caso *OllamaEmbeddings()* para generar el vector
3. Guarda el texto original y su correspondiente vector en la *vector store* de *Chroma*

Podemos ver que los embeddings se estan generando correctamente con el siguiente código (no forma dentro del código final):

In [7]:
data = vector_store.get(include=['embeddings'])

# Para ver los embeddings del primer chunk, no los imprimimos todos (hay 768 xd)
print(data['embeddings'][0][:30])

[ 0.02725698  0.05026644 -0.13049266 -0.07393885  0.03244552 -0.00505756
  0.02465732 -0.00350863 -0.02056746 -0.0068397  -0.01521997 -0.0046853
  0.10491869  0.03440397 -0.01472879  0.01154074  0.01415378 -0.07126766
 -0.00486376  0.01106874  0.03104531 -0.01692409  0.02466148 -0.01964626
  0.01312184  0.00217164  0.01849631 -0.04251862 -0.01740811 -0.00742486]


El *retriever* que se vio anteriormente, se utilizará para el algoritmo de *vector similarity* en nuestra *vector store*, ya que nos proporciona los métodos necesarios para ello.

Podemos ver un ejemplo de su uso, que nuevamente no forma parte del código oficial:

In [8]:
'''query = 'How does RAG compare with fine-tuning?'
relevant_docs = retriever.get_relevant_documents(query)
relevant_docs'''

"query = 'How does RAG compare with fine-tuning?'\nrelevant_docs = retriever.get_relevant_documents(query)\nrelevant_docs"

Y este resultado es la información más relevante de nuestra *vector store* que se asemeja más al prompt. Sin embargo, esto es simplemente un ejemplo muy sencillo, ya que no hemos dado una respuesta, solo la información que es similar. Para ello pasamos a la siguiente sección...

---

### **2. Retriever y Generation**

Los pasos que vamos a seguir en la fase de *retriever* y *generation* son:

- Obtener la user query
- Vectorizar dicha user query
- Hacer un *similarity search* con la *vector store* para encontrar tanto los vectores más relacionados con el input del usuario como su contenido. Esto es de lo que se encarga el **retriever**
- Pasar el contenido obtenido por el *retriever* a un *template*. A esto se conoce como **hydrating**
- Pasar el *hydrated prompt* al LLM
- Presentar la respuesta del LLM al usuario

In [9]:
rag_prompt = hub.pull('jclemens24/rag-prompt')
print(rag_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'jclemens24', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '1a1f3ccb9a5a92363310e3b130843dfb2540239366ebe712ddd94982acc06734'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


Este template lo hemos obtenido directamente de *Langchain Hub*, obteniendo así el prompt que le pasaremos al LLM.

Vemos que este template requiere de un *context* y una *question*, lo cual proporcionaremos más adelante, completando el proceso de **hydrating** correctamente. Este *prompt template* es una parte fundamental del RAG, ya que nos permite comunicarnos con el LLM correctamente. 

No es solo un string, sino un contexto junto con el prompt del usuario para que el LLM proporcione la mejor respuesta posible.

Ahora vamos a definir una función para obtener todo el contenido de los documentos que almacenamos en la *vector store*:

In [10]:
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

Perfecto, ahora el último paso antes de montar la chain (*Langchain chain*) es definir el LLM que vamos a usar:

In [11]:
llm = ChatOllama(model='qwen3:8b', temperature=0)

Ahora vamos a montar la *chain*. Esta *chain* la vamos a definir en un formato en específico llamado LCEL, ya que hace que el código sea más legible y compacto, y abre nuevas técnicas para optimizar la velocidad y eficiencia del código

In [12]:
rag_chain = (
    {'context': retriever | format_docs,
     'question': RunnablePassthrough()}
        | rag_prompt
        | llm
        | StrOutputParser()
)

Esta *chain* representa una cadena de operaciones usando el framework de *Langchain*.

Al poner en uno de los values del dict un `Runnable`, *rag_chain* es un objeto de la clase `RunnableParallel` que devuelve 

    { 'context': <texto>, 'question': <texto> }

donde ambos `context` y `question` reciben el mismo input de la *chain*, pero cada uno lo procesa de forma distinta:

- En **`context`** el input pasa por `retriever | format_docs`, por lo que buscamos los vectores similares en la vector store y los devolvemos como documentos, para pasarlos por la función y obtener el string

- En **`question`** se devuelve el input del usuario sin modificar gracias a `RunnablePassthrough()`

Cabe destacar que `|` no es el típico `OR` de *Python*, sino que conecta el *Runnable* de la izquierda (*retriever*) como input a la función de la derecha (*format_docs*)

---

Al usar el operador `|` estamos construyendo un objeto de la clase **`RunnableSequence`**, donde cada etapa toma la salida de la anterior y la pasa a la siguiente. En concreto:

1. La primera etapa es el **`RunnableParallel`** (el dict), que construye el diccionario `{context, question}`

2. Esa salida entra al **`prompt`** (el cual es un template de Langchain), que inserta `context` y `question` en el template

3. El resultado se envía al **`llm`**, que genera la respuesta

4. Finalmente, **`StrOutputParser()`** pasa la salida del LLM a un string normal y corriente

---

Por lo que solo queda llamar al método `invoke` de *rag_chain* para pasarle el input del usuario y generar la respuesta, pasando por todo el proceso que acabamos de explicar

In [13]:
'''from IPython.display import Markdown

response = rag_chain.invoke('What are the advantajes of using RAG?')

display(Markdown(response))'''

"from IPython.display import Markdown\n\nresponse = rag_chain.invoke('What are the advantajes of using RAG?')\n\ndisplay(Markdown(response))"

---

### Code Lab 3.1. Adding sources to your RAG

Ahora vamos a mejorar el rendimiento del RAG, de la siguiente manera:

In [14]:
from langchain_core.runnables import RunnableParallel

rag_chain_from_docs = (
    RunnablePassthrough.assign(context=(
        lambda x: format_docs(x['context'])
    ))
        | rag_prompt
        | llm
        | StrOutputParser()
)

rag_chain_with_source = RunnableParallel(
    {'context': retriever,
    'question': RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

Vemos que ahora `rag_chain` lo hemos dividido en dos partes:

- **`rag_chain_from_docs`**: Que formatea los documentos recibidos por el contexto y luego insertamos el prompt en el template, enviando la respuesta al llm y transformando el resultado a *string*

- **`rag_chain_with_source`**: El cual se crea usando un `RunnableParallel()` ejecutando *`retriever`* y *`RunnablePassthrough`* en paralelo para obtener `context` y `question` en la *chain*. El resultado se asigna a *`answer`* a través de *`rag_chain_from_docs`*

Esto nos permite separar el *retriever* del *context* del formateo de los documentos scrapeados, lo que nos da una mayor flexibilidad al poder manejar el contexto antes de proporcionarlo al LLM. Finalmente, solo queda proporcionar un *input* a `rag_chain_with_source`:

In [15]:
response_with_source = rag_chain_with_source.invoke('What are the advantages of using RAG?')

In [19]:
from IPython.display import Markdown

print(response_with_source['context'])

display(Markdown(response_with_source['answer']))

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='4. ... 5. ... Constraints:\n1. ~4000 word limit for short term memory. Your short term memory is short, so immediately save important information to files. 2. If you are unsure how you previously did something or want to recall past events, thinking about similar events will help you remember. 3.'), Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='2. Long Term memory management. 3. GPT-3.5 powered Agents for delegation of simple tasks. 4.'), Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='File output. Performance Evaluation:\n1. Continuously review and analyze your actions to ensure you are performing to the best of your abilities. 2. Constructively self-criticize your big-picture behavior constantly. 3. Reflect on past decisions and strategies to refine your approach. 4. Every command

<think>
Okay, the user is asking about the advantages of using RAG. Let me start by recalling what RAG stands for. RAG is Retrieval-Augmented Generation, right? So it combines retrieving information from a database or documents with generating text using a model like GPT.

Looking at the context provided, there are some constraints and performance evaluation points. The context mentions things like short-term memory limits, long-term memory management, using GPT-3.5 agents for tasks, and file output. Also, there's emphasis on efficiency and self-evaluation. 

Wait, the user's question is about RAG advantages, but the context given doesn't directly mention RAG. The context seems to be about system constraints and performance, maybe from a different setup. Since the user provided that context, I need to check if any of it relates to RAG. 

The context talks about saving important info to files (which might relate to memory management), using agents for delegation (maybe similar to how RAG uses retrieval), and efficiency in commands. But none of the context explicitly mentions RAG. The answer should be based on the given context, but if there's no info on RAG, I should say I don't know. 

Wait, the user might have provided the context as part of a different setup. Let me recheck. The context includes points about memory management, agents, file output, and performance evaluation. Maybe these are part of a system that uses RAG, but the context doesn't explicitly state that. Since the question is about RAG advantages and the context doesn't mention RAG, I can't infer the answer from the given context. Therefore, I should respond that I don't know based on the provided information.
</think>

I don't know. The provided context does not mention Retrieval-Augmented Generation (RAG) or its advantages.